In [1]:
import numpy as np
import pandas as pd
import warnings
import numpy as np
import pandas as pd
import numpy as np
import pandas as pd
import altair as alt
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
import torch

warnings.filterwarnings("ignore")
%matplotlib inline
pd.options.display.precision = 15
alt.renderers.enable('mimetype')

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.__version__)
print(torch.version.cuda)


%env JOBLIB_TEMP_FOLDER=/tmp

True
NVIDIA GeForce RTX 3060
2.13.0+cu130
13.0
env: JOBLIB_TEMP_FOLDER=/tmp


In [2]:
folder_path = "dataset/"
train = pd.read_pickle(f"{folder_path}merged_train.pkl")
test = pd.read_pickle(f"{folder_path}merged_test.pkl")

RANDOM_SEED = 42
START_DATE = "2026-01-01"

In [3]:
X = train.drop(columns=["isFraud"])
y = train["isFraud"].values

x = torch.tensor(X.values, dtype=torch.float)
y = torch.tensor(y, dtype=torch.long)

In [4]:
uid_counts = train["uid"].value_counts()

print(uid_counts.describe())
print("Largest uid group:", uid_counts.max())
print(uid_counts.nlargest(20))

count    13657.000000000000000
mean        43.240828878963171
std        324.083169525824928
min          1.000000000000000
25%          1.000000000000000
50%          4.000000000000000
75%         13.000000000000000
max      14112.000000000000000
Name: count, dtype: float64
Largest uid group: 14112
uid
13261    14112
13656    11033
5108     10332
6231     10312
12009     8844
4396      7918
2362      7079
10517     6766
2217      6760
8028      6126
12010     6047
11751     5325
559       5155
2483      5110
8297      4604
7043      4197
4768      3973
5318      3914
8094      3864
5271      3739
Name: count, dtype: int64


In [5]:
import torch

K = 5
edge_list = []

for _, group in train.groupby("uid"):
    group = group.sort_values("TransactionDT")
    idx = group.index.to_list()

    n = len(idx)
    if n < 2:
        continue

    for i in range(n):
        for j in range(i + 1, min(i + K + 1, n)):
            edge_list.append([idx[i], idx[j]])
            edge_list.append([idx[j], idx[i]])

edge_index_uid = torch.tensor(edge_list, dtype=torch.long).t().contiguous()

print(edge_index_uid.shape)

torch.Size([2, 5594098])


In [6]:
import torch

K = 5
edge_list = []

for _, group in train.groupby("uid2"):
    group = group.sort_values("TransactionDT")
    idx = group.index.to_list()

    n = len(idx)
    if n < 2:
        continue

    for i in range(n):
        for j in range(i + 1, min(i + K + 1, n)):
            edge_list.append([idx[i], idx[j]])
            edge_list.append([idx[j], idx[i]])

edge_index_uid2 = torch.tensor(edge_list, dtype=torch.long).t().contiguous()

print(edge_index_uid2.shape)

torch.Size([2, 4610182])


In [7]:
def graph_statistics(edge_index, num_nodes, name):
    degree = torch.bincount(edge_index[0], minlength=num_nodes)

    print(f"===== {name} =====")
    print(f"Nodes           : {num_nodes:,}")
    print(f"Directed edges  : {edge_index.shape[1]:,}")
    print(f"Average degree  : {degree.float().mean():.2f}")
    print(f"Maximum degree  : {degree.max().item():,}")
    print(f"Isolated nodes  : {(degree == 0).sum().item():,}")
    print()

In [8]:
graph_statistics(edge_index_uid, len(train), "G1 (uid)")
graph_statistics(edge_index_uid2, len(train), "G2 (uid2)")

===== G1 (uid) =====
Nodes           : 590,540
Directed edges  : 5,594,098
Average degree  : 9.47
Maximum degree  : 10
Isolated nodes  : 3,473

===== G2 (uid2) =====
Nodes           : 590,540
Directed edges  : 4,610,182
Average degree  : 7.81
Maximum degree  : 10
Isolated nodes  : 36,762



In [ ]:
import torch
from sklearn.model_selection import train_test_split

num_nodes = len(train)

indices = np.arange(num_nodes)

train_idx, valid_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=train["isFraud"]
)

train_mask = torch.zeros(num_nodes, dtype=torch.bool)
valid_mask = torch.zeros(num_nodes, dtype=torch.bool)

train_mask[train_idx] = True
valid_mask[valid_idx] = True

In [ ]:
from torch_geometric.data import Data

data_uid = Data(
    x=x,
    edge_index=edge_index_uid,
    y=y
)

data_uid.train_mask = train_mask
data_uid.val_mask = valid_mask

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data_uid = data_uid.to(device)

In [ ]:
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

class GraphSAGE(torch.nn.Module):
    def __init__(self, in_channels):
        super().__init__()

        self.conv1 = SAGEConv(in_channels, 128)
        self.conv2 = SAGEConv(128, 64)
        self.fc = torch.nn.Linear(64, 2)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        x = self.conv1(x, edge_index)
        x = F.relu(x)

        x = self.conv2(x, edge_index)
        x = F.relu(x)

        x = self.fc(x)

        return x

In [ ]:
model = GraphSAGE(data_uid.num_node_features).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=5e-4
)

criterion = torch.nn.CrossEntropyLoss()

In [ ]:
from sklearn.metrics import roc_auc_score

for epoch in range(1, 21):
    model.train()
    optimizer.zero_grad()
    out = model(data_uid)
    loss = criterion(out[data_uid.train_mask], data_uid.y[data_uid.train_mask])
    loss.backward()
    optimizer.step()
    model.eval()

    with torch.no_grad():
        logits = model(data_uid)
        probs = torch.softmax(logits[data_uid.val_mask], dim=1)[:, 1].cpu().numpy()
        labels = data_uid.y[data_uid.val_mask].cpu().numpy()
        auc = roc_auc_score(labels, probs)

    print(f"Epoch {epoch:02d} " f"Loss={loss:.4f} " f"AUC={auc:.4f}")